# Corpus Statistics

In [2]:
import pandas as pd
import os
import numpy as np

COHFIE_DF = pd.read_csv(f'./COHFIE_V4.csv', chunksize=2000000)

In [3]:
%%time

tokens = pd.Series(dtype='object')
total_tokens = 0
unique_tokens = set()
sentence_lengths = []
filename = f"./COHFIE_STATS.csv"


for data in COHFIE_DF:
    tokens = data['text'].str.replace(r'(?<!\S)XX_[A-Z]*(?!\S)', "").str.replace("[^A-Za-z0-9\s]", "").str.replace("  +", " ").str.strip().str.lower().str.split(' ')
    token_len = tokens.str.len()
    total_tokens += token_len.sum()
    sentence_lengths += token_len.tolist()
    tokens.apply(unique_tokens.update)
    a = data['text']
    b = tokens

    print(f"Total toks: {total_tokens}")
    print(f"Unique toks: {len(unique_tokens)}")
    print(f"Avg. sentence length: {np.mean(sentence_lengths)}")
    
    # Build dataframe
    df = data[['month','year','source_type','source']]
    df['token_len'] = token_len
    
    if os.path.isfile(filename): # if .csv exists
        print(f"Appending to {filename}...")
        df.to_csv(filename, mode='a', index=False, header=False)
    else: # saving for the first time
        print(f"Saving to {filename}...")
        df.to_csv(filename, index=False)

<timed exec>:9: FutureWarning: The default value of regex will change from True to False in a future version.


Total toks: 42789660
Unique toks: 519471
Avg. sentence length: 21.39483
Saving to ./COHFIE_STATS.csv...


<timed exec>:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


Total toks: 86202449
Unique toks: 691150
Avg. sentence length: 21.55061225
Appending to ./COHFIE_STATS.csv...
Total toks: 123123004
Unique toks: 818110
Avg. sentence length: 20.520500666666667
Appending to ./COHFIE_STATS.csv...
Total toks: 147498186
Unique toks: 1092415
Avg. sentence length: 18.43727325
Appending to ./COHFIE_STATS.csv...
Total toks: 169556455
Unique toks: 1346254
Avg. sentence length: 16.9556455
Appending to ./COHFIE_STATS.csv...
Total toks: 191346878
Unique toks: 1557459
Avg. sentence length: 15.945573166666666
Appending to ./COHFIE_STATS.csv...
Total toks: 213471999
Unique toks: 1750967
Avg. sentence length: 15.247999928571428
Appending to ./COHFIE_STATS.csv...
Total toks: 235563425
Unique toks: 1934837
Avg. sentence length: 14.7227140625
Appending to ./COHFIE_STATS.csv...
Total toks: 257690614
Unique toks: 2106597
Avg. sentence length: 14.316145222222222
Appending to ./COHFIE_STATS.csv...
Total toks: 279878611
Unique toks: 2270275
Avg. sentence length: 13.99393055
A

## Table

In [3]:
df = pd.read_csv("./COHFIE_STATS.csv")
df.head()

,month,year,source_type,source,token_len
0,0,1901,books,100year,49
1,0,1901,books,100year,22
2,0,1901,books,100year,24
3,0,1901,books,100year,13
4,0,1901,books,100year,13


In [6]:
df_stats = df.groupby(["source_type", "source"])["source"].agg(sentences=("count"))
df_stats.style.format("{:,.0f}")

### Per source

In [5]:
df_stats = df.groupby(["source_type", "source"])["token_len"].agg(total_tokens=("sum"), mean_sentence_length=("mean"))
df_stats.style.format("{:,.0f}")

### Per source_type

In [7]:
df_stat_per_source = df_stats.groupby("source_type").agg(total_tokens=("total_tokens","sum"), mean_sentence_length=("mean_sentence_length", "mean"))
df_stat_per_source.style.format("{:,.0f}")

,total_tokens,mean_sentence_length
source_type,,
books,"8,907,568",23
news_sites,"94,466,149",20
online_forums,"28,053,083",15
social_media,"792,778,696",12
wikipedia,"9,177,188",32


## Per year

In [8]:
df_stats_per_year = df.groupby(["year"])["token_len"].agg(total_tokens=("sum"), mean_sentence_length=("mean"))
df_stats_per_year.style.format("{:,.0f}")

,total_tokens,mean_sentence_length
year,,
0,"282,690",30
1593,"20,118",19
1870,"46,429",25
1885,"22,806",32
1887,"4,244",56
1888,"3,839",47
1889,"8,050",20
1891,"113,273",26
1898,"12,883",34
